# 🛡️ GraphGuard: Google Drive + Colab Retraining Pipeline

This notebook runs the **GraphGuard (TempoStruct-GAN + GraphSAGE)** training directly using your uploaded folder in **Google Drive**.
- ✅ **Permanent Storage**: All models, plots, and logs save straight to your Google Drive in real-time.
- ✅ **No Data Loss**: Even if Colab disconnects, your trained checkpoints and results remain safely saved in Drive.

---

### Step 1: Set GPU Runtime
> Go to **Runtime** ➔ **Change runtime type** ➔ Select **T4 GPU** ➔ Click **Save**.

In [ ]:
!nvidia-smi

### Step 2: Mount Google Drive
Run this cell and click **Connect to Google Drive** when prompted.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Step 3: Locate Your Uploaded Project Folder in Google Drive
Since you uploaded the folder directly to Drive, this cell automatically detects whether it's named `project1`, `GraphGuard`, etc., and sets the active working directory.

In [ ]:
import os
from pathlib import Path

# Common uploaded folder names
candidate_dirs = [
    Path('/content/drive/MyDrive/project1'),
    Path('/content/drive/MyDrive/GraphGuard'),
    Path('/content/drive/MyDrive/project-1'),
    Path('/content/drive/MyDrive')
]

project_dir = None
for p in candidate_dirs:
    if (p / 'scripts' / '03_train_classifier.py').exists():
        project_dir = p
        break

# If placed in a custom subfolder, scan top 2 levels
if project_dir is None:
    for root, dirs, files in os.walk('/content/drive/MyDrive'):
        if '03_train_classifier.py' in files and 'scripts' in root:
            project_dir = Path(root).parent
            break

if project_dir:
    print(f"✅ Successfully located project folder: {project_dir}\n")
    %cd {str(project_dir)}
    !ls -la
else:
    print("❌ Could not find scripts/03_train_classifier.py in Google Drive!")
    print("Folders in your Google Drive root:")
    !ls -la /content/drive/MyDrive

### Step 4: Install Dependencies

In [ ]:
!pip install -q scikit-learn matplotlib seaborn

### Step 5: Train Generator & Edge Predictor (TempoStruct-GAN)
Trains the temporal generator and edge predictor for 100 epochs.
Checkpoints are saved directly to `graphguard/models/` in your Drive.

In [ ]:
!python scripts/02_train_generator.py

### Step 6: Train Downstream GraphSAGE Classifier
Balances the training set with synthetic illicit nodes, injects time-consistent edges, and trains GraphSAGE for 1000 epochs.
Automatically restores the best checkpoint and evaluates on the test split.

In [ ]:
!python scripts/03_train_classifier.py

### Step 7: View Final Test Metrics & Training Curves
Displays the evaluation metrics and loss/F1 curves saved in your Google Drive.

In [ ]:
import glob, json
from IPython.display import Image, display

runs = sorted(glob.glob('results/classifier_gnn/run_*'))
if runs:
    latest_run = runs[-1]
    print(f"Latest Run: {latest_run}\n")
    
    with open(f"{latest_run}/metrics.json") as f:
        metrics = json.load(f)
        print("=" * 40)
        print("FINAL TEST METRICS:")
        print("=" * 40)
        for k, v in metrics.items():
            print(f"{k:25s}: {v}")
        print("=" * 40)
        
    print("\n--- Validation F1 Curve ---")
    display(Image(f"{latest_run}/val_f1_curve.png"))
    print("\n--- Loss Curve ---")
    display(Image(f"{latest_run}/loss_curve.png"))
else:
    print("No classifier runs found yet. Run Step 6 first!")